# Sahel Rainfall Trends — ICON vs IFS-FESOM (1990–2050)

**Bounding box:** North Africa  lat: [-10, 25], lon: [-20, 55]

**Plots:**
1. 2040s mean minus 1990s mean rainfall (mm/day)
2. Linear trend in rainfall over 1990–2050 (mm/day total change)

Each plot has two subplots: left = ICON, right = IFS-FESOM.
Coastlines only — no borders, no gridlines.

**Prerequisites:** `conda activate destine`, `~/.polytopeapirc` configured.

In [ ]:
# Install earthkit-plots if not already present (needed for Map + coastlines)
%pip install earthkit-plots -q

In [ ]:
# ── Lightweight imports + logging suppression ─────────────────────
import logging, warnings

for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import sys, os, pathlib
import numpy as np
import pandas as pd
import xarray as xr

print("imports OK")

In [ ]:
# ── Add get-data/ to sys.path so polytope_zarr can be imported ──
_nb_dir = pathlib.Path(os.getcwd())
_proj_root = _nb_dir.parent if _nb_dir.name == "analysis" else _nb_dir
_get_data = str(_proj_root / "get-data")
if _get_data not in sys.path:
    sys.path.insert(0, _get_data)
print("sys.path includes:", _get_data)

## 1. Define bounding box & create stores (lazy — no download yet)

(Imports `earthkit.data` and `PolytopeZarrStore` — may pause briefly.)

In [ ]:
# ── Bounding box (south, west, north, east) ──────────────────────
BBOX = (-10, -20, 25, 55)   # North Africa / Sahel region
MODELS = ["ICON", "IFS-FESOM"]
VAR = "avg_tprate"          # Time-mean total precipitation rate (kg m⁻² s⁻¹)
KG_M2S_TO_MMDAY = 86400.0   # 1 kg m⁻² s⁻¹ = 1 mm/s → ×86400 = mm/day

In [ ]:
import earthkit.data
from polytope_zarr import PolytopeZarrStore

# ── Historical store  (1990–2014) ────────────────────────────────
hist_store = PolytopeZarrStore.from_climate_dt(
    models=MODELS,
    experiment="hist",
    levtype="sfc",
    frequency="monthly",
    years=range(1990, 2015),
    resolution="standard",
)
print("hist store:", hist_store)

In [ ]:
# ── Scenario store   (2015–2050) ─────────────────────────────────
scen_store = PolytopeZarrStore.from_climate_dt(
    models=MODELS,
    experiment="SSP3-7.0",
    levtype="sfc",
    frequency="monthly",
    years=range(2015, 2051),
    resolution="standard",
)
print("scen store:", scen_store)

## 2. Open lazy datasets & verify connectivity

In [ ]:
# Open as lazy xarray Datasets (instant — no data fetched yet)
ds_hist = hist_store.open()
ds_scen = scen_store.open()
print("hist ds dims:", dict(ds_hist.sizes))
print("scen ds dims:", dict(ds_scen.sizes))

In [ ]:
# Verify stores can connect (one small request each)
print("=== Verify hist store ===")
r_hist = hist_store.verify()
print(r_hist["message"])

print("\n=== Verify scen store ===")
r_scen = scen_store.verify()
print(r_scen["message"])

## 3. Fetch data via bounding-box feature requests

The `.polytope.sel(bbox=...)` accessor triggers a **server-side** Polytope
feature request, returning data regridded to a lat/lon grid for the
specified bounding box.

**Timing note:** each call below fetches data from the remote Polytope
server and may take 30–120 seconds depending on the time range.

In [ ]:
def fetch_bbox(ds, model, time_slice, label=""):
    """Fetch a bounding-box subset from a lazy xarray Dataset.

    Parameters
    ----------
    ds : xr.Dataset
        Lazy dataset opened via ``store.open()``.
    model : str
        Model name ("ICON" or "IFS-FESOM").
    time_slice : slice
        e.g. ``slice("1990-01", "1999-12")``.
    label : str
        Human-readable label for progress messages.

    Returns
    -------
    xr.DataArray
        Precipitation rate on a lat/lon grid with dimensions
        (time, latitude, longitude) in kg m⁻² s⁻¹.
    """
    print(f"  Fetching {model} {label} ... ", end="", flush=True)
    result_ds = ds.polytope.sel(
        var=VAR,
        model=model,
        time=time_slice,
        bbox=BBOX,
    )
    var_names = list(result_ds.data_vars)
    if len(var_names) == 0:
        raise KeyError(f"No data variables returned for {model} {label}")
    da = result_ds[var_names[0]]
    print(f"done — shape {dict(da.sizes)}, var={var_names[0]}")
    if "time" in da.coords and not np.issubdtype(da["time"].dtype, np.datetime64):
        da["time"] = pd.to_datetime(da["time"].values)
    return da

### 3a. Fetch 1990s (historical) & 2040s (scenario) for change plot

In [ ]:
# ICON — 1990s (1990-1999) & last year of scenario (2040)
icon_1990s = fetch_bbox(ds_hist, "ICON", slice("1990-01", "1999-12"), "1990s")
icon_2040s = fetch_bbox(ds_scen, "ICON", slice("2040-01", "2040-12"), "2040s")

In [ ]:
# IFS-FESOM — 1990s (1990-1999) & last decade of scenario (2040-2049)
fesom_1990s = fetch_bbox(ds_hist, "IFS-FESOM", slice("1990-01", "1999-12"), "1990s")
fesom_2040s = fetch_bbox(ds_scen, "IFS-FESOM", slice("2040-01", "2049-12"), "2040s")

### 3b. Fetch full 1990–2050 for trend plot

In [ ]:
# ICON — full historical + scenario (1990–2040)
icon_hist_full = fetch_bbox(ds_hist, "ICON", slice("1990-01", "2014-12"), "hist 1990-2014")
icon_scen_full = fetch_bbox(ds_scen, "ICON", slice("2015-01", "2040-12"), "SSP3-7.0 2015-2040")
icon_full = xr.concat([icon_hist_full, icon_scen_full], dim="time")

In [ ]:
# IFS-FESOM — full historical + scenario (1990–2049)
fesom_hist_full = fetch_bbox(ds_hist, "IFS-FESOM", slice("1990-01", "2014-12"), "hist 1990-2014")
fesom_scen_full = fetch_bbox(ds_scen, "IFS-FESOM", slice("2015-01", "2049-12"), "SSP3-7.0 2015-2049")
fesom_full = xr.concat([fesom_hist_full, fesom_scen_full], dim="time")

## 4. Compute statistics

In [ ]:
# ── Decadal means (mm/day) ───────────────────────────────────────
icon_1990s_mean  = icon_1990s.mean("time")  * KG_M2S_TO_MMDAY
icon_2040s_mean  = icon_2040s.mean("time")  * KG_M2S_TO_MMDAY
fesom_1990s_mean = fesom_1990s.mean("time") * KG_M2S_TO_MMDAY
fesom_2040s_mean = fesom_2040s.mean("time") * KG_M2S_TO_MMDAY

# ── Change signals (mm/day) ──────────────────────────────────────
icon_change  = icon_2040s_mean  - icon_1990s_mean
fesom_change = fesom_2040s_mean - fesom_1990s_mean

print("ICON 1990s mean range:",  icon_1990s_mean.min().values,  "–", icon_1990s_mean.max().values)
print("ICON change range:",      icon_change.min().values,      "–", icon_change.max().values)
print("FESOM 1990s mean range:", fesom_1990s_mean.min().values, "–", fesom_1990s_mean.max().values)
print("FESOM change range:",     fesom_change.min().values,     "–", fesom_change.max().values)

In [ ]:
def linear_trend(da):
    """Compute linear trend (total change over period) per grid cell.

    Fits a least-squares line via numpy.polyfit to annual-mean
    values, then multiplies by the number of years so the result
    is the *total change* in mm/day over the entire period.

    Parameters
    ----------
    da : xr.DataArray
        Monthly data with dims (time, latitude, longitude)
        in kg m⁻² s⁻¹.

    Returns
    -------
    xr.DataArray
        Total change in mm/day over the period, dims
        (latitude, longitude).
    """
    annual = da.resample(time="YS").mean("time")
    annual_mmday = annual * KG_M2S_TO_MMDAY
    n_years = annual_mmday.sizes["time"]

    t = annual_mmday.time.dt.year.values.astype(float)
    t = t - t.mean()

    def _trend(y):
        mask = np.isfinite(y)
        if mask.sum() < 2:
            return np.nan
        slope, _ = np.polyfit(t[mask], y[mask], 1)
        return slope

    slope = xr.apply_ufunc(
        _trend, annual_mmday,
        input_core_dims=[["time"]],
        vectorize=True,
        dask="allowed",
        output_dtypes=[float],
    )
    total_change = slope * (n_years - 1)
    total_change.name = "trend"
    return total_change


# ── Compute trends (total change in mm/day over each model's full period) ─
icon_trend  = linear_trend(icon_full)
fesom_trend = linear_trend(fesom_full)

print("ICON trend range:",  icon_trend.min().values,  "–", icon_trend.max().values,  "mm/day")
print("FESOM trend range:", fesom_trend.min().values, "–", fesom_trend.max().values, "mm/day")

## 5. Plot 1 — 2040s mean minus 1990s mean rainfall (mm/day)

**Left:** ICON  |  **Right:** IFS-FESOM

Coastlines only (no borders, no gridlines).

In [ ]:
import earthkit.plots  # loaded here to avoid early import hangs

# ── ICON change (left) ───────────────────────────────────────────
chart_left = earthkit.plots.Map(domain=[-10, 25, -20, 55])
chart_left.point_cloud(icon_change, x="longitude", y="latitude")
chart_left.coastlines()
chart_left.title("ICON — 2040s − 1990s")

# ── IFS-FESOM change (right) ─────────────────────────────────────
chart_right = earthkit.plots.Map(domain=[-10, 25, -20, 55])
chart_right.point_cloud(fesom_change, x="longitude", y="latitude")
chart_right.coastlines()
chart_right.title("IFS-FESOM — 2040s − 1990s")

chart_left.show()
chart_right.show()

## 6. Plot 2 — Linear trend in rainfall 1990–2050 (mm/day total change)

Total change = slope × (N−1) years from least-squares fit to annual means.

**Left:** ICON (1990–2040)  |  **Right:** IFS-FESOM (1990–2049)

Coastlines only (no borders, no gridlines).

In [ ]:
# ── ICON trend (left) ────────────────────────────────────────────
chart_left = earthkit.plots.Map(domain=[-10, 25, -20, 55])
chart_left.point_cloud(icon_trend, x="longitude", y="latitude")
chart_left.coastlines()
chart_left.title("ICON — linear trend 1990–2040")

# ── IFS-FESOM trend (right) ──────────────────────────────────────
chart_right = earthkit.plots.Map(domain=[-10, 25, -20, 55])
chart_right.point_cloud(fesom_trend, x="longitude", y="latitude")
chart_right.coastlines()
chart_right.title("IFS-FESOM — linear trend 1990–2049")

chart_left.show()
chart_right.show()

In [ ]:
# Free memory if needed
hist_store.clear_cache()
scen_store.clear_cache()